In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class CustomImageDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_id = os.listdir(img_dir)

        self.transform = transform

    def __len__(self):
        return len(self.img_id)

    def __getitem__(self, idx):
        img_path =  os.path.join(self.img_dir, self.img_id[idx])
        mask_path = os.path.join(self.mask_dir, self.img_id[idx])

        # Load image
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path)

        # Convert to black and white
        thresh = 128
        fn = lambda x: 255 if x > thresh else 0
        mask = mask.convert('L').point(fn,mode='1')

        if self.transform:
            image, mask = self.transform(image, mask)
        return image, mask

In [ ]:
import torchvision.transforms.v2 as transforms
from segmentation_models_pytorch import utils
import segmentation_models_pytorch as smp
import torch


transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    ])

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# load best saved checkpoint
best_model = torch.load('./best_model.pth')

img_dir = './RetinaBloodVessel/test/image'
mask_dir = './RetinaBloodVessel/test/mask'

test_dataset = CustomImageDataset(img_dir, mask_dir, transform=transform)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

loss = utils.losses.DiceLoss()
metrics = [
    utils.metrics.IoU(threshold=0.5),
]

# evaluate model on test set
test_epoch = smp.utils.train.ValidEpoch(
    model=best_model,
    loss=loss,
    metrics=metrics,
    device=DEVICE
)

logs = test_epoch.run(test_dataloader)

valid: 100%|██████████| 20/20 [00:02<00:00,  8.55it/s, dice_loss - 0.3089, iou_score - 0.5455]
